In [ ]:
import torch
import numpy as np
import time

In [ ]:
# Generate random test inputs
num_samples = 1000
X = torch.rand((num_samples), dtype=torch.float32)
R = torch.rand((num_samples), dtype=torch.float32)
ones = torch.ones((num_samples), dtype=torch.float32)
dataset = torch.stack([torch.log10(X), torch.log10(R), torch.log10(X)*torch.log10(R)], dim=1)

In [ ]:
from pynq import get_rails, DataRecorder

rails = get_rails()
recorder = DataRecorder(rails['12V'].power)

# Test CPU

In [ ]:
from espertaTorch import MultiEsperta
# Configuration matching original models
ESPERTA_CONFIGS = [
    # s1 models
    ([-6.07, -1.75, 1.14, 0.56], 0.28),  # w20_120
    ([-7.44, -2.99, 1.21, 0.69], 0.28),  # e40_19
    ([-5.02, -1.74, 0.64, 0.40], 0.23),  # e120_41
    
    # s2 models
    ([-6.07, -1.75, 1.14, 0.56], 0.35),  # w20_120
    ([-7.44, -2.99, 1.21, 0.69], 0.28),  # e40_19
    ([-5.02, -1.74, 0.64, 0.40], 0.23),  # e120_41
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultiEsperta(ESPERTA_CONFIGS)

In [ ]:
recorder.record(0.1)
time.sleep(5)

In [ ]:
recorder.mark()

In [ ]:
# Get outputs from PyTorch model
cpu_inference_time = time.time()
pytorch_outputs = model(dataset).detach().numpy()
cpu_inference_time = time.time() - cpu_inference_time

avg_inference_time = cpu_inference_time / num_samples
fps = num_samples / cpu_inference_time

print(f"The total inference time was {cpu_inference_time:.6f} seconds for {num_samples}.")
print("Average inference time per image: {:.6f} seconds".format(avg_inference_time))
print("FPS: {:.2f}".format(fps))

# Test DPU

In [ ]:
# VitisAI does not support the following operations:
# DPU does not support sigmoid
# aten::gt can't be converted to XIR.

# Test HLS

In [ ]:
recorder.mark()

In [ ]:
from pynq import Overlay
ol = Overlay("hls_bitstream/ESPERTA.bit")
ESPERTA_ip = ol.entry_0

In [ ]:
print(ESPERTA_ip.register_map)
def load_input_to_ESPERTA(input_data):
    input_data_np = input_data.numpy()
    ESPERTA_ip.write(0x10, input_data_np[0].tobytes())
    ESPERTA_ip.write(0x14, input_data_np[1].tobytes())
    ESPERTA_ip.write(0x18, input_data_np[2].tobytes())

def get_ESPERTA_output():
    output_1 = ESPERTA_ip.read(0x20).to_bytes(4, 'little')
    output_2 = ESPERTA_ip.read(0x24).to_bytes(4, 'little')
    ESPERTA_0 = output_1[0]
    ESPERTA_1 = output_1[1]
    ESPERTA_2 = output_1[2]
    ESPERTA_3 = output_1[3]
    ESPERTA_4 = output_2[0]
    ESPERTA_5 = output_2[1]
    return ESPERTA_0, ESPERTA_1, ESPERTA_2, ESPERTA_3, ESPERTA_4, ESPERTA_5

def start_ESPERTA():
    ESPERTA_ip.write(0x00, 0x01)
    while ESPERTA_ip.read(0x00) & 0x4 == 0:
        continue

def run_ESPERTA(input_data):
    load_input_to_ESPERTA(input_data)
    start_ESPERTA()
    return get_ESPERTA_output()

In [ ]:
hls_inference_time = 0
hls_outputs = []
for data in dataset:
    load_input_to_ESPERTA(data)
    start_time = time.time()
    start_ESPERTA()
    hls_inference_time += time.time() - start_time
    out_data = get_ESPERTA_output()
    hls_outputs.append(out_data)

# Convert list to numpy array
hls_outputs = np.array(hls_outputs)

avg_infer_time = hls_inference_time/num_samples
fps = num_samples/hls_inference_time
print(f"The total inference time was {hls_inference_time:.6f} seconds for {num_samples}.")
print("Average inference time: {:.6f} s".format(avg_infer_time))
print("Performance: {:.2f} FPS".format(fps))

In [ ]:
recorder.stop()

In [ ]:
model_names = [
    "S1 W20_120",
    "S1 E40_19",
    "S1 E120_41",
    "S2 W20_120",
    "S2 E40_19",
    "S2 E120_41"
]

# Compare outputs using MSE and check if they're the same
mse = np.mean((pytorch_outputs - hls_outputs)**2, axis=0)
is_same = np.allclose(pytorch_outputs, hls_outputs, rtol=1e-5, atol=1e-5)

print("Comparison of PyTorch vs Original models:")
print("=" * 50)
print(f"Are all outputs the same? {'Yes' if is_same else 'No'}")
print("-" * 50)
speedup = cpu_inference_time / hls_inference_time
slowdown = hls_inference_time / cpu_inference_time
print(f"Speedup of HLS over CPU: {speedup:.2f}x")
print(f"Slow-down of HLS over CPU: {slowdown:.2f}x")
for i, name in enumerate(model_names):
    print(f"{name}:")
    print(f"  Mean Square Error: {mse[i]:.8e}")
    is_model_same = np.allclose(pytorch_outputs[:, i], hls_outputs[:, i], rtol=1e-5, atol=1e-5)
    print(f"  Outputs match: {'Yes' if is_model_same else 'No'}")

In [ ]:
import matplotlib.pyplot as plt

# Get the timestamps of the marks
mark_indices = []
last_mark = 0.0
for i, mark in enumerate(recorder.frame['Mark']):
    if mark > last_mark:
        mark_indices.append(i)
        last_mark = mark

# Extract the power data
power_data = recorder.frame['12V_power']
time_data = recorder.frame.index

# Create the plot
plt.figure(figsize=(12, 6))
plt.plot(time_data, power_data, color='gray')

# Color the plot based on the marks
plt.plot(time_data[mark_indices[2]:], power_data[mark_indices[2]:], color='orange', label='FPGA Inference')
plt.plot(time_data[:mark_indices[1]], power_data[:mark_indices[1]], color='blue', label='CPU Inference')

# Add labels and title with larger font size
plt.xlabel('Time', fontsize=14)
plt.ylabel('Power (W)', fontsize=14)
plt.title('Power Consumption of CPU and FPGA Inference for MMS Neural Networks', fontsize=12)

# Add legend with larger font size
plt.legend(fontsize=16)

# Increase tick label font size
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

# Show the plot
plt.show()